In [1]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (313/313), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 313 (delta 159), reused 241 (delta 95), pack-reused 0 (from 0)
Receiving objects: 100% (313/313), 1.18 MiB | 5.83 MiB/s, done.
Resolving deltas: 100% (159/159), done.


In [2]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 381.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 275.2 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [3]:
!pip install -q stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 24.2 MB/s eta 0:00:00


In [4]:
from typing import Optional
from enum import Enum
from logging import Logger

import cv2
import torch
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import KVWriter, Logger as PpoLogger

from mario_the_explorer import (SuperMarioWorldLayeredEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger,
                                tile_absolute_id, TileEncoder, SuperMarioAction, SuperMarioCombo, SuperMarioDiscretizer,
                                prime_policy_for_combo, TileType, SCREEN_COLUMNS, SCREEN_ROWS, TILE_SIZE)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
class Direction(Enum):
    LEFT = 0
    RIGHT = 1
    UP = 2
    DOWN = 3

In [7]:
from re import T
from mario_the_explorer.environment import tiles
class TryThingsRewardModel(RewardModel):
    def __init__(self, new_block_reward = 10.0, action_reward = 1.0, progression_reward = 0.1):
        self._blocks_seen = set()
        self._block_action_counts = {}
        self._block_available_rewards = {}
        self._best_distance_reached = 0
        self._new_block_reward = new_block_reward
        self._action_reward = action_reward
        self._progression_reward = progression_reward
        self.MAX_BLOCK_REWARD = self._get_total_reward_for_block()

    def reset(self) -> None:
        self._blocks_seen = set()
        self._block_action_counts = {}
        self._block_available_rewards = {}
        self._best_distance_reached = 0

    def _get_total_reward_for_block(self, number_of_actions = 10, number_of_directions = 4) -> float:
        total_possible_reward = 0.0
        last_total_possible_reward = -1
        interactions = 0
        while total_possible_reward != last_total_possible_reward:
            last_total_possible_reward = total_possible_reward
            interactions += 1
            total_possible_reward += self._reward_curve(interactions)
        return total_possible_reward * number_of_actions * number_of_directions

    def _reward_curve(self, action_count: int) -> float:
        action_reward = self._action_reward / action_count
        if action_reward < 0.05:
            action_reward = 0.0
        return action_reward

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        reward = 0.0
        for row in observation:
            for tile in row:
                tile_id = tile_absolute_id(tile)
                if tile["type"] == TileType.EMPTY:
                    self._block_available_rewards[tile_id] = 0
                    continue
                if tile["type"] == TileType.MARIO:
                    self._block_available_rewards[tile_id] = 0
                    continue
                if tile_id not in self._blocks_seen:
                    self._blocks_seen.add(tile_id)
                    reward += 10.0
                    self._block_available_rewards[tile_id] = self.MAX_BLOCK_REWARD
        tiles_around_mario = self._get_tiles_around_mario(observation)
        for tile_and_direction in tiles_around_mario:
            combo_id = SuperMarioCombo.get_combo_id_from_action(action)
            if combo_id == SuperMarioCombo.DO_NOTHING:
                continue
            interaction_id = (combo_id, tile_and_direction[0], tile_and_direction[1])
            action_reward = 0.0
            if interaction_id not in self._block_action_counts:
                self._block_action_counts[interaction_id] = 0
            self._block_action_counts[interaction_id] += 1
            action_reward = self._reward_curve(self._block_action_counts[interaction_id])
            self._block_available_rewards[tile_and_direction[1]] -= action_reward
            reward += action_reward
        if info["x"] > self._best_distance_reached:
            self._best_distance_reached = info["x"]
            reward += 0.1
        if terminated:
            unique_tiles_visible = set()
            for row in observation:
                for tile in row:
                    unique_tiles_visible.add(tile_absolute_id(tile))
            valid_tiles_visible = len(unique_tiles_visible) - 2
            max_reward_possible = valid_tiles_visible * self.MAX_BLOCK_REWARD
            reward_still_available = 0
            for tile_id in self._block_available_rewards:
                reward_still_available += self._block_available_rewards[tile_id]
            explored_percentage = reward_still_available / max_reward_possible
            reward -= explored_percentage * 20.0
        return reward

    def _get_tiles_around_mario(self, observation: list[list[Tile]]) -> set[tuple[Direction, int]]:
        mario_coordinates = self._find_mario_coordinates(observation)
        blocks_around_mario = set()
        if not mario_coordinates:
            return blocks_around_mario
        for mario_row, mario_col in mario_coordinates:
            if mario_row > 0:
                block_above_mario = observation[mario_row - 1][mario_col]
                if block_above_mario["type"] != TileType.MARIO and block_above_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.UP.name, tile_absolute_id(block_above_mario)))
            if mario_row < len(observation) - 1:
                block_below_mario = observation[mario_row + 1][mario_col]
                if block_below_mario["type"] != TileType.MARIO and block_below_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.DOWN.name, tile_absolute_id(block_below_mario)))
            if mario_col > 0:
                block_left_of_mario = observation[mario_row][mario_col - 1]
                if block_left_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.LEFT.name, tile_absolute_id(block_left_of_mario)))
            if mario_col < len(observation[0]) - 1:
                block_right_of_mario = observation[mario_row][mario_col + 1]
                if block_right_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.RIGHT.name, tile_absolute_id(block_right_of_mario)))
        return blocks_around_mario


    def _find_mario_coordinates(self, observation: list[list[Tile]]) -> list[tuple[int, int]]:
        mario_coordinates = []
        for row_idx, row in enumerate(observation):
            for col_idx, tile in enumerate(row):
                if tile["type"] == TileType.MARIO:
                    mario_coordinates.append((row_idx, col_idx))
        return mario_coordinates

    def get_potential_map(self, observation: list[list[Tile]]) -> np.ndarray:
        h, w = len(observation), len(observation[0])
        potential_map = np.zeros((h, w), dtype=np.float32)
        for r in range(h):
            for c in range(w):
                t_id = tile_absolute_id(observation[r][c])
                potential_map[r, c] = self._block_available_rewards.get(t_id, 0.0)/self.MAX_BLOCK_REWARD
        return potential_map

In [8]:
class MarioReshapeWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # Get the original H, W (14, 16)
        c, h, w = self.observation_space.shape
        # Redefine the space to include a single channel (1, 14, 16)
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(c+1, h, w),
            dtype=np.float32
        )

    def observation(self, obs):
        reward_model = self.env.unwrapped.reward_model
        heatmap = reward_model.get_potential_map(self.env.unwrapped.observation)
        heatmap = np.expand_dims(heatmap, axis=0)
        screen = np.concat([obs, heatmap], axis=0).astype(np.float32)
        return screen

In [9]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import torch.nn as nn

class CustomMarioCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 512):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten()
        )

        # Compute shape by doing one forward pass
        with torch.no_grad():
            sample_tensor = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample_tensor).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU(),
            nn.Linear(features_dim, features_dim),
            nn.ReLU()
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.linear(self.cnn(observations))

In [10]:
GRID_COLOR_RGB = (80, 80, 80)

class RewardHeatmapOverlay(ScreenOverlay):
    def __init__(self, reward_model: TryThingsRewardModel):
        self._reward_model = reward_model
        self.img_width = SCREEN_COLUMNS * TILE_SIZE
        self.img_height = SCREEN_ROWS * TILE_SIZE

    def apply(self, original_frame: np.ndarray, observation: Optional[list[list[Tile]]]):
        if observation is None:
            return original_frame
        reward_map = self._reward_model.get_potential_map(observation)
        reward_image = self._get_reward_image(reward_map)
        return np.vstack((original_frame, reward_image))

    def _get_reward_image(self, reward_map):
        matrix_img = np.zeros((self.img_height, self.img_width, 3), dtype=np.uint8)
        matrix_img = self._populate_objects(matrix_img, reward_map)
        matrix_img = self._draw_grid(matrix_img)
        unused_region = np.zeros((self.img_height, self.img_width, 3), dtype=np.uint8)
        return np.hstack((matrix_img, unused_region))

    def _populate_objects(self, matrix_img, reward_map):
        for row in range(SCREEN_ROWS):
            for col in range(SCREEN_COLUMNS):
                tile_available_reward = reward_map[row][col]
                self._draw_tile(matrix_img, col, row, tile_available_reward)
        return matrix_img

    def _draw_tile(self, img, x: int, y: int, tile_available_reward: float) -> None:
        scaled_reward = int(tile_available_reward * 255)
        visually_uniform_grayscale = scaled_reward ** 2.2
        color = (visually_uniform_grayscale, visually_uniform_grayscale, visually_uniform_grayscale)
        x_start = int(x * TILE_SIZE)
        y_start = int(y * TILE_SIZE)
        x_end = x_start + TILE_SIZE - 1
        y_end = y_start + TILE_SIZE - 1
        cv2.rectangle(img, (x_start, y_start), (x_end, y_end), color, -1)

    def _draw_grid(self, img):
        h, w = img.shape[:2]
        for x in range(0, w + 1, TILE_SIZE):
            cv2.line(img, (x, 0), (x, h), GRID_COLOR_RGB, 1)
        for y in range(0, h + 1, TILE_SIZE):
            cv2.line(img, (0, y), (w, y), GRID_COLOR_RGB, 1)
        return img

In [11]:
RUN_NAME = "larger_policy_network"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"
MAX_STEPS = 2000

In [12]:
class PpoKvWriter(KVWriter):
    def __init__(self, logger: Logger):
        self._logger = logger

    def write(self, key_values, key_excluded, step=0):
        for key, value in key_values.items():
            self._logger.info(f"Step {step} - {key}: {value}")

    def close(self):
        pass

In [13]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
ppo_logger = PpoLogger(
    folder=None,
    output_formats=[PpoKvWriter(logger)]
)
reward_model = TryThingsRewardModel()
overlay = RewardHeatmapOverlay(reward_model)
base_env = SuperMarioWorldLayeredEmulator(level = LEVEL,
                                          render_mode = "rgb_array",
                                          reward_model = reward_model,
                                          screen_overlay = overlay,
                                          render_debug = True,
                                          render_grid = True,
                                          max_episode_length = MAX_STEPS,
                                          limit_fps_to = 5,
                                          logger = logger)
env = SuperMarioDiscretizer(base_env)
env = MarioReshapeWrapper(env)

2026-05-07 13:21:56 [INFO] Session log for run larger_policy_network with level [INFO] initialized at: larger_policy_network_20260507_132156.log


In [14]:
try:
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=512),
    )
    model = PPO("CnnPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=500, device=device)
    model.set_logger(ppo_logger)
    # prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=30000)
    model.save("ppo_mario_test")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

Using cpu device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 64, but because the `RolloutBuffer` is of size `n_steps * n_envs = 500`, after every 7 untruncated mini-batches, there will be a truncated mini-batch of size 52
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=500 and n_envs=1)
  warnings.warn(
2026-05-07 13:22:03 [INFO] Starting training...
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
2026-05-07 13:22:14 [INFO] Step 500 - time/iterations: 1
2026-05-07 13:22:14 [INFO] Step 500 - time/fps: 42
2026-05-07 13:22:14 [INFO] Step 500 - time/time_elapsed: 11
2026-05-07 13:22:14 [INFO] Step 500 - time/total_ti

In [15]:
from gymnasium.wrappers import RecordVideo

try:
    env.env.env.limit_fps_to = None
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-07 13:40:19 [INFO] Terminated: True
2026-05-07 13:40:19 [INFO] Truncated: False
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
try:
    env.env.env.limit_fps_to = 5
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=512),
    )
    model = PPO("CnnPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=2000, device=device)
    model.set_logger(ppo_logger)
    # prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=500000)
    model.save("ppo_mario_trained")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

/usr/local/lib/python3.12/dist-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 64, but because the `RolloutBuffer` is of size `n_steps * n_envs = 2000`, after every 31 untruncated mini-batches, there will be a truncated mini-batch of size 16
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=2000 and n_envs=1)
  warnings.warn(


Using cpu device


2026-05-07 13:40:21 [INFO] Starting training...
2026-05-07 13:41:09 [INFO] Step 2000 - train/learning_rate: 0.0003
2026-05-07 13:41:09 [INFO] Step 2000 - train/entropy_loss: -1.34400622099638
2026-05-07 13:41:09 [INFO] Step 2000 - train/policy_gradient_loss: -0.0028627907664940722
2026-05-07 13:41:09 [INFO] Step 2000 - train/value_loss: 2427.255340576172
2026-05-07 13:41:09 [INFO] Step 2000 - train/approx_kl: 0.008335994556546211
2026-05-07 13:41:09 [INFO] Step 2000 - train/clip_fraction: 0.010501802968792617
2026-05-07 13:41:09 [INFO] Step 2000 - train/loss: 1138.121337890625
2026-05-07 13:41:09 [INFO] Step 2000 - train/explained_variance: 0.0
2026-05-07 13:41:09 [INFO] Step 2000 - train/n_updates: 600
2026-05-07 13:41:09 [INFO] Step 2000 - train/clip_range: 0.2
2026-05-07 13:41:09 [INFO] Step 2000 - time/iterations: 1
2026-05-07 13:41:09 [INFO] Step 2000 - time/fps: 41
2026-05-07 13:41:09 [INFO] Step 2000 - time/time_elapsed: 48
2026-05-07 13:41:09 [INFO] Step 2000 - time/total_times

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    env.env.env.limit_fps_to = None
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"trained-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

In [ ]:
try:
    env.env.env.limit_fps_to = 5
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=512),
    )
    model = PPO.load("ppo_mario_trained")
    model.set_logger(ppo_logger)
    # prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=5000000)
    model.save("ppo_mario_full_train")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    env.env.env.limit_fps_to = None
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"full-train-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    done = False
    step_count = 0
    while not done:
        env.render()
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        step_count += 1
        if step_count % 1000 == 0:
            logger.info(f"Step: {step_count}")
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

In [ ]:
try:
    env.env.env.limit_fps_to = 5
    ppo_env = DummyVecEnv([lambda: env])
    policy_kwargs = dict(
        features_extractor_class=CustomMarioCNN,
        features_extractor_kwargs=dict(features_dim=512),
    )
    model = PPO("CnnPolicy", ppo_env, policy_kwargs=policy_kwargs, verbose=1, learning_rate=0.0003, n_steps=3000, device=device)
    model.set_logger(ppo_logger)
    # prime_policy_for_combo(model, SuperMarioCombo.RIGHT_RUN, ppo_env, logger, iterations=1)
    logger.info("Starting training...")
    model.learn(total_timesteps=50000000)
    model.save("ppo_mario_full_train")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.reset()